In [ ]:
import torch
import functools
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer, DPOTrainer
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel
from peft import LoraConfig
from huggingface_hub import login
from dotenv import load_dotenv
import os

In [ ]:
class TrainPipelineDPO:
    def __init__(self):
        pass
    def hf_login(self, token):
        login(token)
    def get_dataset(self, path):
        ds = Dataset.from_json(path)
        return ds
    def preprocess_function_dpo(self, example, prompt_col="prompt", chosen_col="chosen", rejected_col="rejected"):
        return {
            "prompt": [{"role": "user", "content": example["prompt"]}],
            "chosen": [{"role": "assistant", "content": example["chosen"]}],
            "rejected": [{"role": "assistant", "content": example["rejected"]}]
        }


    def prepare_model(self, model_name, token):
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            token=token,
            cache_dir="./model_cache"
        )
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=token,
            cache_dir="./model_cache",
            quantization_config=bnb_config,
            device_map="auto",
            low_cpu_mem_usage=True,
            attn_implementation="eager"
            # attn_implementation="flash_attention_2"
        )

        base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

        model = PeftModel.from_pretrained(
            base_model,
            "drive/MyDrive/outputFixed/checkpoint-3169", # This is where your SFT adapters live
            is_trainable=True # Important for DPO
        )



        model.config.use_cache = False

        lora = LoraConfig(
            r=16,
            lora_alpha=32,
            # r=8,          # reduce from 16
            # lora_alpha=16, # reduce from 32
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            task_type="CAUSAL_LM"
        )
        from trl import DPOTrainer, DPOConfig

        training_args = DPOConfig(
            output_dir="drive/MyDrive/output_gemma_dpo",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            bf16=True,
            learning_rate=5e-7,
            lr_scheduler_type="cosine",
            warmup_steps=50,
            num_train_epochs=1,
            logging_steps=10,
            save_steps=10,
            save_total_limit=5,
            gradient_checkpointing=True,
            max_length=1024,
            beta=0.1,
            report_to="none",
            remove_unused_columns=False,
            )


        training_args.require_safe_serialization = False

        return model, tokenizer, lora, training_args

    def delete_model(self, *args):
        for obj in args:
            del obj
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**2:.1f} MB")
        print("Model deleted from RAM.")




In [ ]:
trainPipeline = TrainPipelineDPO()

In [ ]:
load_dotenv()
HF_HUB_TOKEN = os.getenv("HF_HUB_TOKEN")
trainPipeline.hf_login(HF_HUB_TOKEN)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset_path = r"drive/MyDrive/human-Like-DPO-Dataset-mk-train.jsonl"

ds = trainPipeline.get_dataset(dataset_path)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
formatted_dataset = ds.map(
    trainPipeline.preprocess_function_dpo,
    batched=False,
    remove_columns=ds.column_names,
    load_from_cache_file=False
)

Parameter 'function'=<bound method TrainPipelineDPO.preprocess_function_dpo of <__main__.TrainPipelineDPO object at 0x7d55b1fb0d40>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/8707 [00:00<?, ? examples/s]

In [ ]:
model_name = "google/gemma-3-1b-it"
# model_name = "google/gemma-3-4b-it"

model, tokenizer, lora, training_args = trainPipeline.prepare_model(model_name, HF_HUB_TOKEN)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

trainer = DPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=formatted_dataset
)

logging.disable(logging.NOTSET)

Tokenizing train dataset:   0%|          | 0/8707 [00:00<?, ? examples/s]

In [ ]:
original_log = DPOTrainer.log

def patched_log(self, logs, start_time=None):
    return original_log(self, logs)

DPOTrainer.log = patched_log

# trainer.train()

In [ ]:
trainer.train(resume_from_checkpoint=True)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Step,Training Loss
400,0.699971
410,0.688774
420,0.696121
430,0.700130
440,0.693391
450,0.694021
460,0.698435
470,0.696901
480,0.689540
490,0.699388


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

TrainOutput(global_step=545, training_loss=0.1977413339352389, metrics={'train_runtime': 12949.5053, 'train_samples_per_second': 0.672, 'train_steps_per_second': 0.042, 'total_flos': 3.948122650207334e+16, 'train_loss': 0.1977413339352389})